In [1]:
import torch

In [2]:
print("torch version:", torch.__version__)
print("cuda build:", torch.version.cuda)
print("is available:", torch.cuda.is_available())

torch version: 2.5.1+cu121
cuda build: 12.1
is available: True


In [5]:
import cv2
import os

path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"

print("Path:", path)
print("Exists:", os.path.exists(path))

Path: d:\ANLAM_Lab\⚒️MediaEval2026\media_code_NEW\.venv\Lib\site-packages\cv2\data\haarcascade_frontalface_default.xml
Exists: True


In [4]:
import cv2

path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"

cascade = cv2.CascadeClassifier()
cascade.load(path)

print("Loaded:", not cascade.empty())

Loaded: False


In [6]:
import shutil
import cv2
import os

src = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
dst = "C:/temp/haarcascade_frontalface_default.xml"

os.makedirs("C:/temp", exist_ok=True)
shutil.copy(src, dst)

FACE_CASCADE = cv2.CascadeClassifier(dst)

print("Loaded:", not FACE_CASCADE.empty())

Loaded: True


checking if load frames (<60) for a video

In [1]:
from pathlib import Path
vid_id = "ZwkM_C_y_Pc"
frame_dir = Path("frames") / vid_id

In [2]:
# sorted frames → take first MAX_FRAMES
MAX_FRAMES = 60
frame_files = sorted(frame_dir.iterdir())[:MAX_FRAMES]

print("Frame files:", frame_files)

Frame files: [WindowsPath('frames/ZwkM_C_y_Pc/000001.jpg'), WindowsPath('frames/ZwkM_C_y_Pc/000031.jpg'), WindowsPath('frames/ZwkM_C_y_Pc/000061.jpg'), WindowsPath('frames/ZwkM_C_y_Pc/000091.jpg'), WindowsPath('frames/ZwkM_C_y_Pc/000121.jpg'), WindowsPath('frames/ZwkM_C_y_Pc/000151.jpg')]


load csv, group by `channelName` and print the `id` of the given `channelName`

In [7]:
# load csv, group by `channelName` and print the `id` of the given `channelName`s.
import pandas as pd

df = pd.read_csv("devset_videolist_GT.csv")
grouped = df.groupby("channelName")

channel_names = ["HSBC UK", "Natixis Investment Managers US", "BNP Paribas"]
for channel_name in channel_names:
    if channel_name in grouped.groups:
        ids = grouped.get_group(channel_name)["id"].tolist()
        print(f"IDs for {channel_name}: {ids}")
    else:
        print(f"{channel_name} not found in the dataset.")


IDs for HSBC UK: ['NxELm2V4K3o', 'v52sVQVyeV0', 'Sruprfe-ilA', 'sIcISsIQO4g', 'xrs9WVjMuSA', 'MDeGdAykBzI', '_6awP7nueuM', '6PfweQbwWkQ', 'ZkJH0ZdYyag', 'EAkOEhFhstM', 'si-SY6aEPb0', 'ZwkM_C_y_Pc']
IDs for Natixis Investment Managers US: ['y9EveQ1sO8g', 'JO2zOqN4Zu4', 'B_kUZ7Nwafs']
IDs for BNP Paribas: ['Sq10VripUxw', 'IWf7AieBQZY', '9xJesiLceew', 'Kl3NWKgiKfg', 'TDKOsv9s6t0', '3Rwvbbp4Uz4']


In [8]:
# load csv, group by `channelName` and print the `id` of the given `channelName`s.

df = pd.read_csv("testset_videolist_.csv")
grouped = df.groupby("channelName")

print("=== Test Set ===")
channel_names = ["HSBC", "Natixis Investment Managers", "BNP Paribas Asset Management"]
for channel_name in channel_names:
    if channel_name in grouped.groups:
        ids = grouped.get_group(channel_name)["id"].tolist()
        print(f"IDs for {channel_name}: {ids}")
    else:
        print(f"{channel_name} not found in the dataset.")


=== Test Set ===
IDs for HSBC: ['Ex41SlPPVms', 'IpoNLcMo130', 'MLGCHY35IqA', 'MjAblRlVIV4', 'XLKCBTYv79U', 'XYfG3y__Nk4', 'S5XsfgPkZBE', 'jrC9GXqTolM', '0C8S3rk1uxg', 'NLMJql1tSJw', 'hzwyv8tOj74', 'BQnw_5crCqk', '7wC9zKp8kMw', '7kSay5gWXv0', 'xG8HMmj4rek', 'ACIavgaYwJs']
IDs for Natixis Investment Managers: ['PESufOQXpCY', 'C_4MpbDvZjI', 'adk2eYE1SFw']
IDs for BNP Paribas Asset Management: ['7rQYL07Dw0k', 'HPYR9JCRTCU', 'IAFMhrXJ_OA', 'cTXC-MN4PGk', 'kkhjAM1Uz4I', 'doFUQ8vDSko', '5OczIrSM6y0']


if any test channels overlap with the 32 training channels

In [10]:
import pandas as pd
train = pd.read_csv("devset_videolist_GT.csv")
test  = pd.read_csv("testset_videolist_.csv")

train_channels = set(train["channelName"].unique())
test_channels  = set(test["channelName"].unique())

print("Test channels seen in training:", test_channels & train_channels)
print("Test channels unseen:          ", test_channels - train_channels)
print(f"\n{len(test_channels & train_channels)}/{len(test_channels)} test channels have training data")

Test channels seen in training: set()
Test channels unseen:           {'Aberdeen Standard Investments', 'BNP Paribas Asset Management', 'FranklinTempletonTV', 'Natixis Investment Managers', 'Schroders', 'BlackRock', 'Fidelity Investments', 'Nomura', 'HSBC'}

0/9 test channels have training data


Explore Goldman Sachs videos — print titles and basic stats to inform how to split into subchannels.

In [1]:
"""
Usage:
    python explore_gs.py --csv devset_videolist_GT.csv --stt devset-stt/
"""
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

def main():
    df   = pd.read_csv("devset_videolist_GT.csv")
    gs   = df[df["channelName"] == "Goldman Sachs"].copy()

    print(f"Goldman Sachs: {len(gs)} videos")
    print(f"  mem  — mean={gs['memorability_score'].mean():.3f}  std={gs['memorability_score'].std():.3f}")
    print(f"  brand — mean={gs['brand_memorability'].mean():.3f}  std={gs['brand_memorability'].std():.3f}")
    print(f"  duration — min={gs['durationSeconds'].min()}s  max={gs['durationSeconds'].max()}s  "
          f"median={gs['durationSeconds'].median():.0f}s")
    print(f"  categories: {gs['categoryName'].value_counts().to_dict()}")

    print(f"\n{'─'*80}")
    print(f"{'title':<60}  {'mem':>6}  {'brand':>6}  {'dur':>6}s")
    print(f"{'─'*80}")
    for _, row in gs.sort_values("memorability_score", ascending=False).iterrows():
        print(f"{str(row['title'])[:58]:<60}  {row['memorability_score']:.3f}  "
              f"{row['brand_memorability']:.3f}  {row['durationSeconds']:>6}")

    # simple keyword-based topic detection from titles
    print(f"\n{'─'*80}")
    print("Keyword frequency in titles (to inform cluster labels):")
    all_words = " ".join(gs["title"].str.lower().fillna("")).split()
    from collections import Counter
    counts = Counter(all_words)
    # filter short/common words
    ignore = {"the","a","an","and","of","in","for","to","with","on","at","by","our","we","how","what","why","is","are","has","have","you","your","can","|","&","-","–"}
    filtered = {w: c for w, c in counts.items() if w not in ignore and len(w) > 3 and c >= 2}
    for word, count in sorted(filtered.items(), key=lambda x: -x[1])[:30]:
        print(f"  {word:<25} {count}")


if __name__ == "__main__":
    main()

Goldman Sachs: 79 videos
  mem  — mean=0.773  std=0.116
  brand — mean=0.574  std=0.144
  duration — min=30s  max=3647s  median=917s
  categories: {'News & Politics': 79}

────────────────────────────────────────────────────────────────────────────────
title                                                            mem   brand     durs
────────────────────────────────────────────────────────────────────────────────
Orlando Bravo, Managing Partner of Thoma Bravo                1.000  0.400     628
Phasing in the Reopening of the Economy                       0.944  0.722    3647
The Daily Check-In: How the Pandemic is Reshaping Educatio    0.933  0.800     526
10,000 Small Businesses – Long Live Small Businesses          0.929  0.571      30
Jenny Reardon: The Science and Ethics of Genomics             0.929  0.786    1461
Jean Case: Author, ''Be Fearless''                            0.900  0.600    1307
Michael Dell, Chairman and CEO of Dell Technologies           0.895  0.421    1332